# Lección 14.2 — Exploración vs. explotación: ¿lo de siempre o algo nuevo?

> **Módulo 14 · Aprendizaje por refuerzo (introducción conceptual)** · Nivel 5 — Héroe
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 14.1 (agente, entorno, recompensa)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar el dilema **exploración vs. explotación** con una analogía cotidiana.
- Describir cómo funciona la estrategia **ε-greedy** (épsilon-codiciosa) en un problema de *bandidos multibrazo*.
- Predecir el efecto de los hiperparámetros **ε (epsilon)**, su **decaimiento** y el **factor de descuento γ (gamma)**.
- Comparar estrategias de exploración con simulaciones y elegir una para un caso de negocio.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: ¿tu restaurante favorito o uno nuevo?

Es viernes y vas a cenar. Tienes un restaurante favorito que **nunca falla**. Pero en tu barrio abrieron tres nuevos.

- Si vas **siempre** a tu favorito, nunca sabrás si alguno de los nuevos es aún mejor. → **Explotar** (*exploit*): aprovechar lo que ya sabes.
- Si **siempre** pruebas uno nuevo, cenarás mal muchas veces aunque ya conozcas uno bueno. → **Explorar** (*explore*): reunir información.

Ninguno de los extremos es bueno. La gracia está en **el equilibrio**. Este es el dilema central del aprendizaje por refuerzo: recuerda que el agente solo aprende de lo que **prueba** (Lección 14.1).

### ✋ Ejercicio 1 — Calentamiento
¿En cuál de estas dos situaciones explorarías **más**?
- a) Te mudaste a una ciudad nueva donde vivirás 5 años.
- b) Estás de paso en una ciudad, solo por una noche.

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

En **a)**. Con 5 años por delante, lo que descubras explorando lo vas a **aprovechar muchas veces**. Con una sola noche, conviene ir a lo seguro (explotar).

Regla general: **cuanto más largo es el horizonte, más vale la pena explorar al principio.** Esta idea reaparecerá con el factor de descuento γ en la Sección 5.
</details>

---
## 2. 📖 El concepto: el bandido multibrazo 🎰

El problema más simple donde aparece el dilema es el **bandido multibrazo** (*multi-armed bandit*). El nombre viene de las tragamonedas, a las que en inglés llaman "bandidos de un brazo" (te quitan el dinero con una palanca).

Tienes **5 tragamonedas**. Cada una paga un premio (recompensa = 1) con una probabilidad distinta, que **tú no conoces**. Tienes 1,000 jugadas. ¿Cómo ganar lo máximo posible?

```
   🎰 A      🎰 B      🎰 C      🎰 D      🎰 E
   25 %?     45 %?     60 %?     35 %?     75 %?     ← secreto: el agente no lo sabe
```

Es RL en miniatura: hay **acciones** (elegir máquina) y **recompensas**, pero no hay estados que cambien. Aparece en la vida real cada vez que eliges entre opciones repetidas: qué anuncio mostrar, qué versión de una página web, qué tratamiento en un ensayo clínico.

### 📖 ¿Cómo estima el agente qué máquina es mejor?
Lleva un registro y calcula el **promedio de premios** de cada máquina con las veces que la ha jugado. A ese promedio se le llama **valor estimado** de la acción, y se suele escribir **Q** (de *quality*, calidad). Lo verás mucho en la Lección 14.3.

### ✋ Ejercicio 2 — Estima a mano
Tras 10 jugadas, este es tu historial:

| Jugada | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| Máquina | A | B | A | C | B | B | A | C | B | B |
| Premio | 1 | 0 | 0 | 1 | 1 | 1 | 0 | 0 | 0 | 1 |

1. Calcula el valor estimado Q de A, B y C.
2. Si ahora tuvieras que **explotar**, ¿cuál jugarías?
3. ¿Qué tan seguro estás de esa estimación?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Q(A) = 1/3 ≈ **0.33** · Q(B) = 3/5 = **0.60** · Q(C) = 1/2 = **0.50**.
2. La **B**.
3. Muy poco seguro: con 2 a 5 jugadas por máquina, el azar pesa muchísimo. C podría ser en realidad la mejor, y las máquinas D y E **ni siquiera se han probado**. Por eso hay que seguir explorando.
</details>

▶️ La celda siguiente define las 5 máquinas y un agente que juega con la estrategia **ε-greedy**, que verás a continuación. Por ahora solo ejecútala.

In [ ]:
PROBABILIDADES = np.array([0.25, 0.45, 0.60, 0.35, 0.75])   # el secreto del casino
NOMBRES = list("ABCDE")

def jugar(epsilon, n_jugadas=1000, q_inicial=0.0, decaimiento=1.0, probs=PROBABILIDADES, semilla=0):
    """Un agente ε-greedy juega n_jugadas. Devuelve qué eligió, qué ganó y sus estimaciones Q."""
    rng = np.random.default_rng(semilla)
    k = len(probs)
    Q = np.full(k, q_inicial, dtype=float)   # valor estimado de cada máquina
    N = np.zeros(k)                          # veces que jugó cada una
    elecciones, premios = np.zeros(n_jugadas, int), np.zeros(n_jugadas)
    for t in range(n_jugadas):
        if rng.random() < epsilon:                          # 🎲 EXPLORAR: máquina al azar
            a = rng.integers(k)
        else:                                               # 💰 EXPLOTAR: la mejor según Q
            a = rng.choice(np.flatnonzero(Q == Q.max()))    #    (si hay empate, una al azar)
        premio = float(rng.random() < probs[a])
        N[a] += 1
        Q[a] += (premio - Q[a]) / (N[a] + 1)                # actualiza el promedio (el valor inicial
                                                            # cuenta como una "jugada imaginaria")
        elecciones[t], premios[t] = a, premio
        epsilon *= decaimiento                              # (opcional) explorar cada vez menos
    return elecciones, premios, Q, N

def resumen(elecciones, premios, Q, N):
    for i, nombre in enumerate(NOMBRES):
        print(f"🎰 {nombre}: jugada {int(N[i]):>4} veces · Q estimado = {Q[i]:.2f} · real = {PROBABILIDADES[i]:.2f}")
    print(f"Premios totales: {int(premios.sum())} de {len(premios)} jugadas")

---
## 3. 📖 Cómo funciona: de "codicioso" a "ε-greedy"

### El agente codicioso (*greedy*)
La estrategia más ingenua: **siempre** juega la máquina con mayor Q. Es pura explotación (ε = 0).

### 🔮 Ejercicio 3 — El agente codicioso
Todas las Q empiezan en 0. ¿Qué crees que pasará? ¿Encontrará la máquina **E** (la mejor, 75 %)?

✍️ *Mi predicción:*

In [ ]:
resumen(*jugar(epsilon=0.0, semilla=0))

<details><summary><b>👀 Ver respuesta</b></summary>

Se queda **atascado en la C** las 1,000 jugadas y nunca prueba ni la D ni la E. Al principio todas valen 0; eligió la C por sorteo, ganó un premio, su Q pasó a ser mayor que 0 y desde entonces **siempre** le parece la mejor. Como nunca vuelve a jugar las demás, nunca corrige su error.

Ganó unos 590 premios. Con la E habría ganado unos 750. **Explotar sin explorar te deja atrapado en lo "suficientemente bueno".**
</details>

### 📖 La estrategia ε-greedy
Una solución sencilla y sorprendentemente eficaz. En cada jugada, tira un dado:
- con probabilidad **ε** (por ejemplo, 10 %) → **explora**: elige una máquina **al azar**;
- el resto del tiempo (90 %) → **explota**: elige la de mayor Q.

```
     ¿número al azar < ε?
        ├── Sí (10 %) → 🎲 máquina al azar
        └── No (90 %) → 💰 máquina con mayor Q
```

### 🔮 Ejercicio 4 — Un poco de curiosidad
Con ε = 0.1, ¿descubrirá la máquina E? ¿Cuántas veces (aprox.) jugará cada una de las **otras**? (Pista: el 10 % de 1,000 jugadas se reparte al azar entre 5 máquinas.)

✍️ *Mi predicción:*

In [ ]:
resumen(*jugar(epsilon=0.1, semilla=0))

<details><summary><b>👀 Ver respuesta</b></summary>

Sí: juega la **E** la gran mayoría de las veces (más de 800). Las demás reciben unas **20 jugadas** cada una (≈ 100 jugadas de exploración ÷ 5 máquinas), justo lo que predice la pista. Esa exploración "desperdiciada" es el **precio** de no quedarse atascado.

Fíjate también en las estimaciones: la Q de la E queda muy cerca de 0.75 (tiene muchos datos), mientras que las Q de las otras son más imprecisas (pocos datos). **No hace falta conocer bien a las malas: basta con saber que son peores.**
</details>

---
## 4. 📖 ¿Qué pasa si exploras demasiado (o demasiado poco)?

Un solo juego puede ser suerte. Para comparar en serio, repetimos el experimento **300 veces** con cada ε y promediamos (como en la Lección 14.1).

### 🔮 Ejercicio 5 — ¿Qué ε gana?
Compararemos ε = 0 · 0.01 · 0.1 · 0.3 · 1.0 (este último explora siempre: juega totalmente al azar).
1. ¿Cuál ganará más premios en promedio?
2. ¿Qué premio promedio por jugada esperas para ε = 1.0? (Pista: promedio de las 5 probabilidades.)

✍️ *Mis predicciones:*

In [ ]:
EPSILONS = [0.0, 0.01, 0.1, 0.3, 1.0]     # ✏️ prueba otros valores

fig, ax = plt.subplots()
for eps in EPSILONS:
    resultados = [jugar(eps, semilla=s) for s in range(300)]
    acierto_en_t = np.mean([e == 4 for e, _, _, _ in resultados], axis=0)   # % de veces que eligió la E en cada jugada
    premio_medio = np.mean([p.mean() for _, p, _, _ in resultados])
    ax.plot(acierto_en_t, label=f"ε = {eps}")
    print(f"ε = {eps:<5} → premio medio por jugada = {premio_medio:.3f} · eligió la mejor el {acierto_en_t.mean():.0%} del tiempo")
ax.set_xlabel("Jugada"); ax.set_ylabel("% de agentes que eligen la mejor máquina")
ax.legend(); ax.grid(alpha=.3); ax.set_title("¿Cuánto explorar?"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. Gana **ε = 0.1** (≈ 0.69 por jugada).
2. Con ε = 1.0 el premio medio es ≈ **0.48**, el promedio de las 5 máquinas: nunca aprovecha lo que aprende.

Lee la gráfica:
- **ε = 0** (codicioso): se estanca muy pronto en ~30 %. Muchos agentes quedan atascados para siempre en una máquina mediocre.
- **ε = 0.01**: sube, pero **muy despacio**: explora tan poco que tarda cientos de jugadas en descubrir la E.
- **ε = 0.1**: sube rápido y se estabiliza alrededor de 80–90 %.
- **ε = 0.3**: aprende rápido, pero se estanca más abajo: sigue desperdiciando el 30 % de las jugadas en exploración aunque ya sabe cuál es la mejor.
- **ε = 1.0**: plano en 20 % (1 de 5): pura exploración.

Es la misma curva en "U" del sesgo-varianza (Lección 4.5), ahora con el eje "cuánto explorar".
</details>

---
## 5. 📖 Hiperparámetros: las perillas de la curiosidad

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `epsilon` (ε) | Fracción de acciones al azar | Descubre antes, pero desperdicia más jugadas | Aprovecha más, pero puede quedarse atascado | 0.1 (o empezar en 1.0 y decaer) |
| `decaimiento` de ε | Qué tan rápido baja ε con el tiempo | Más cerca de 1 → baja lento: explora más tiempo | Baja rápido: deja de explorar antes de tiempo | Llegar a ε ≈ 0.01–0.05 a mitad o al final del entrenamiento |
| `q_inicial` (valor inicial optimista) | Qué tan buenas "cree" que son las opciones no probadas | Más optimista: prueba todo al principio | 0: no hay incentivo para probar | 0, o algo mayor que la mejor recompensa posible |
| `gamma` (γ) | Cuánto valen las recompensas futuras | Más paciente: planea a largo plazo, aprende más lento | Más miope: solo le importa lo inmediato | 0.9–0.99 |

### 📖 Explorar mucho al principio y poco al final
Lo lógico es explorar **al principio**, cuando no sabes nada, y explotar **al final**, cuando ya aprendiste. Eso hace el **decaimiento**: en cada jugada, ε se multiplica por un número un poco menor que 1.

### ✏️ Ejercicio 6 — ε que decae
Empezamos con ε = 1.0 (todo exploración) y lo multiplicamos por `DECAIMIENTO` en cada jugada. Con 0.99, tras 100 jugadas ε vale 0.99¹⁰⁰ ≈ 0.37 y tras 500, ≈ 0.007.

¿Superará al mejor ε fijo (0.1, ≈ 0.70)? Prueba también 0.9 (decae muy rápido) y 0.999 (decae muy lento).

✍️ *Mi predicción:*

In [ ]:
DECAIMIENTO = 0.99    # ✏️ prueba 0.9, 0.99 y 0.999

resultados = [jugar(1.0, decaimiento=DECAIMIENTO, semilla=s) for s in range(300)]
print(f"Decaimiento {DECAIMIENTO} → premio medio por jugada = {np.mean([p.mean() for _, p, _, _ in resultados]):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

- **0.99** → ≈ **0.71**: le gana (por poco) al mejor ε fijo. Explora mucho al principio y casi nada al final.
- **0.9** → ε se hace casi 0 en unas 40 jugadas: deja de explorar **demasiado pronto** y a veces se atasca (≈ 0.66).
- **0.999** → sigue explorando mucho durante casi todo el juego: desperdicia jugadas (≈ 0.58).

El decaimiento es una perilla más que hay que ajustar, pero la idea "explorar mucho al principio y poco al final" casi siempre ayuda.
</details>

### ✏️ Ejercicio 7 — Optimismo ante la incertidumbre
Otra forma de explorar **sin azar**: empezar con todas las Q en un valor **optimista**, por ejemplo 1.0 (mejor que cualquier máquina real). El agente codicioso prueba una máquina, se "decepciona" (su Q baja) y pasa a la siguiente, que todavía parece perfecta… hasta que las prueba todas.

Ejecuta el agente **codicioso** (ε = 0) con `q_inicial=1.0`. ¿Cuántas veces juega cada máquina? Compáralo con el Ejercicio 3.

✍️ *Mi observación:*

In [ ]:
Q_INICIAL = 1.0    # ✏️ compara con 0.0
resumen(*jugar(epsilon=0.0, q_inicial=Q_INICIAL, semilla=0))

<details><summary><b>👀 Ver respuesta</b></summary>

Con `q_inicial = 1.0` el codicioso **prueba todas las máquinas** al principio y termina jugando sobre todo la **E**. Con 0.0 se atascaba en la C.

Si lo repites 300 veces, el codicioso optimista gana ≈ 0.71 por jugada en promedio: muchísimo mejor que el codicioso ingenuo (≈ 0.54) y, en este problema, incluso mejor que ε-greedy con ε = 0.1 (≈ 0.69), porque no desperdicia jugadas al azar una vez que ya probó todo. Su punto débil: deja de explorar en cuanto se "decepciona" de todas, así que no sirve si el mundo cambia (Ejercicio 9). Otra técnica famosa basada en el mismo optimismo es **UCB** (*Upper Confidence Bound*), que da un "bono" a las opciones poco probadas.
</details>

### 📖 El factor de descuento γ: ¿premio cercano o premio lejano?
En la Lección 14.1 viste que el **retorno descontado** multiplica cada recompensa futura por γ una vez por paso de espera. γ decide **qué tan paciente** es el agente.

Volvamos al pasillo con dos quesos: el **chico** (+2) está a **1 paso** y el **grande** (+10) a **5 pasos**. Sin costo por paso, el valor de cada opción es:

```
queso chico  = 2                  (llega en el primer paso: no se descuenta)
queso grande = γ⁴ · 10            (se descuenta 4 veces: hay 4 pasos de espera)
```

### 🔮 Ejercicio 8 — ¿Qué γ hace falta para ser paciente?
¿A partir de qué γ el agente preferirá el queso grande? Haz tu apuesta: ¿0.3, 0.5, 0.7 o 0.9?

✍️ *Mi apuesta:*

In [ ]:
gammas = np.linspace(0, 1, 101)
valor_chico  = np.full_like(gammas, 2.0)
valor_grande = gammas**4 * 10

plt.plot(gammas, valor_chico, label="Queso chico (+2, a 1 paso)")
plt.plot(gammas, valor_grande, label="Queso grande (+10, a 5 pasos)")
cruce = gammas[np.argmax(valor_grande > valor_chico)]
plt.axvline(cruce, ls="--", c="gray")
plt.xlabel("γ (gamma)"); plt.ylabel("Valor para el agente"); plt.legend(); plt.grid(alpha=.3)
plt.title(f"El agente prefiere el queso grande cuando γ > {cruce:.2f}"); plt.show()

for g in [0.5, 0.9, 0.99]:
    print(f"γ = {g:<5} → horizonte aproximado ≈ {1 / (1 - g):.0f} pasos")

<details><summary><b>👀 Ver respuesta</b></summary>

A partir de γ ≈ **0.67**. Con γ = 0.5 el queso grande vale 0.5⁴·10 ≈ 0.6 (¡menos que el chico!); con γ = 0.9 vale ≈ 6.6.

La última parte muestra una regla práctica: el agente "mira hacia adelante" unos **1/(1 − γ)** pasos. Con γ = 0.9 son ~10 pasos; con γ = 0.99, ~100. Si tu tarea necesita planear 50 pasos (por ejemplo, salir de un laberinto grande), un γ de 0.9 se queda corto.

> ⚖️ ¿Por qué no usar siempre γ = 0.999? Porque cuanto más lejos mira el agente, más **incierto y lento** es aprender: hay que esperar mucho para saber si una acción fue buena (el problema de asignación de crédito de la Lección 14.1).
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| El agente repite siempre la misma acción mediocre | Sin exploración (ε = 0) o ε decae demasiado rápido | Subir ε, decaer más lento, valores iniciales optimistas |
| El aprendizaje mejora pero se estanca por debajo de lo posible | ε fijo demasiado alto: sigue explorando cuando ya sabe | Decaer ε con el tiempo |
| Al evaluar el agente "final" sigue haciendo cosas raras | Se evalúa con ε > 0 | Evaluar con ε = 0 (solo explotar) |
| El agente ignora metas lejanas | γ demasiado bajo | Subir γ (0.95–0.99) |
| El aprendizaje es lentísimo e inestable | γ demasiado cerca de 1 en tareas largas | Bajar γ un poco, recompensas intermedias |
| El mundo cambió y el agente no se entera | Dejó de explorar (ε ≈ 0) en un entorno que cambia | Mantener un ε mínimo (por ejemplo, 0.01–0.05) |

### 🔮 Ejercicio 9 — El casino cambia las máquinas
Imagina que, tras 1,000 jugadas, el casino cambia en secreto la máquina A y ahora paga el **95 %**. Compara dos agentes que llevan 1,000 jugadas: uno con ε fijo = 0.1 y otro con ε decayendo (que ahora vale casi 0). ¿Cuál se dará cuenta del cambio? ¿Por qué?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

El de **ε fijo = 0.1**. Sigue probando la A de vez en cuando (unas 20 veces cada 1,000 jugadas), así que su Q de la A irá subiendo y terminará cambiándose a ella. El de ε ≈ 0 ya no explora: **nunca** volverá a la A y no se enterará.

Esto conecta con producción (Lección 15.2): cuando el mundo cambia (*deriva*), un sistema que dejó de explorar o de medir **no se da cuenta**.
</details>

---
## 7. 📋 Ficha del modelo

| | Agente ε-greedy |
|---|---|
| **Tipo** | Estrategia de exploración para RL (bandidos y, como verás, Q-learning) |
| **Intuición** | "Casi siempre voy a mi restaurante favorito, pero de vez en cuando pruebo uno al azar" |
| **Cómo aprende** | Estima el valor Q de cada acción promediando sus recompensas; elige la mejor con probabilidad 1 − ε y una al azar con probabilidad ε |
| **Hiperparámetros clave** | `epsilon`, `decaimiento`, (`q_inicial`), y `gamma` cuando hay varios pasos |
| **✅ Fortalezas** | Simplísimo · Funciona sorprendentemente bien · Fácil de explicar |
| **❌ Debilidades** | Explora "a ciegas" (prueba igual de seguido una opción pésima que una casi buena) · Hay que ajustar ε |
| **Úsalo cuando…** | Necesitas una línea base de exploración rápida y robusta |
| **Alternativas** | Valores optimistas, UCB, *Thompson sampling* (exploran de forma más inteligente) |

## 8. 🧠 Resumen en 5 puntos
1. Un agente solo aprende de lo que prueba: debe equilibrar **explorar** (reunir información) y **explotar** (aprovechar lo que sabe).
2. El **bandido multibrazo** es la versión más simple del dilema, y aparece en anuncios, pruebas A/B y ensayos clínicos.
3. **ε-greedy** explora al azar con probabilidad ε; demasiado poco ε → se atasca; demasiado ε → desperdicia.
4. **Explorar mucho al principio y poco al final** (decaimiento de ε) suele funcionar mejor que un ε fijo.
5. **γ** decide qué tan paciente es el agente: mira hacia adelante unos 1/(1 − γ) pasos.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** Un agente con ε = 0 desde el principio…
- a) Explora todo · b) Puede quedarse atascado en una opción mediocre · c) Siempre encuentra la mejor opción · d) Juega al azar

**2.** Con ε = 0.2 y 5 acciones, ¿con qué probabilidad **total** elige el agente la acción que cree mejor? (Ojo: al explorar también puede tocarle por azar.)
- a) 0.80 · b) 0.84 · c) 0.20 · d) 1.00

**3.** Para que un agente valore una recompensa que llegará dentro de ~100 pasos, conviene un γ de…
- a) 0.5 · b) 0.9 · c) 0.99 · d) 0

**4.** ¿Por qué se evalúa al agente final con ε = 0?
- a) Para que entrene más rápido · b) Para medir lo que aprendió, sin las acciones al azar de la exploración · c) Porque ε solo sirve en bandidos · d) No se hace

**5.** Un valor inicial de Q **optimista** sirve para…
- a) Que el agente explore todas las opciones al principio · b) Que aprenda más lento · c) Bajar γ · d) Evitar el *reward hacking*

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Como en el Ejercicio 3.
2. **b)** 0.80 por explotar + 0.20 × 1/5 = 0.04 de que la exploración al azar caiga justo en ella → 0.84.
3. **c)** Horizonte ≈ 1/(1 − 0.99) = 100 pasos.
4. **b)**
5. **a)** Ejercicio 7.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Un hospital prueba 3 tratamientos para una enfermedad y quiere usar un bandido para asignar pacientes. ¿Qué tiene de delicado **explorar** en este contexto? ¿Qué cambiarías respecto al casino?

**B.** Una prueba A/B clásica (Lección 4.6 aplicada a negocio) reparte el tráfico **50/50** entre dos versiones de una página durante dos semanas y luego elige la ganadora. Explícala con el vocabulario de esta lección: ¿cuándo explora y cuándo explota? ¿Qué ventaja tendría un bandido?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Cada "exploración" es un paciente que recibe un tratamiento que podría ser peor. El costo de explorar es humano, no unas monedas. Se usan diseños más cuidadosos: explorar lo **mínimo** necesario, estrategias más inteligentes que el azar (Thompson sampling), reglas para **detener** temprano un tratamiento claramente peor, supervisión ética y regulatoria.

**B.** La prueba A/B **explora al 100 %** durante dos semanas (reparte por igual, sin importar cuál va ganando) y luego **explota al 100 %**. Un bandido va desplazando el tráfico hacia la mejor versión **mientras** aprende, así que pierde menos clientes con la versión mala. A cambio, la A/B da una medición más limpia y fácil de analizar con estadística clásica. Lo comprobarás en la práctica integradora.
</details>

### 9.3 🧩 Práctica integradora: "¿Qué anuncio mostramos?"

Una tienda en línea tiene **4 anuncios** para su página principal y recibirá **20,000 visitas** este mes. Cada anuncio tiene una tasa de clic (CTR) desconocida. El equipo de marketing propone tres estrategias:

1. **Prueba A/B/C/D:** las primeras 4,000 visitas se reparten por igual entre los 4 anuncios; después, se muestra **siempre** el que tuvo más clics.
2. **ε-greedy** con ε = 0.1 fijo.
3. **ε-greedy con decaimiento** (empieza en 1.0).

Lo harás en 5 pasos cortos. **No avances sin revisar cada respuesta.**

**Paso 1 ▶️ — El mundo (secreto) y un "oráculo"**

In [ ]:
CTR = np.array([0.030, 0.050, 0.045, 0.020])    # secreto: el anuncio B (índice 1) es el mejor
VISITAS = 20_000

clics_oraculo = VISITAS * CTR.max()               # lo que lograría alguien que ya supiera la respuesta
print(f"Un oráculo que siempre muestra el mejor anuncio lograría ≈ {clics_oraculo:.0f} clics")

**Paso 2 🔮 — La prueba A/B/C/D.** Antes de ejecutar: durante las 4,000 visitas de prueba, ¿qué CTR medio obtendrá la tienda? (Promedio de los 4 CTR.) ¿Elegirá siempre el anuncio correcto (B)? Ojo: B (5.0 %) y C (4.5 %) están muy cerca.

✍️ *Mi predicción:*

In [ ]:
def prueba_ab(ctr, visitas, n_prueba=4000, semilla=0):
    rng = np.random.default_rng(semilla)
    k = len(ctr)
    anuncio = np.arange(n_prueba) % k                      # reparto por turnos durante la prueba
    clics_prueba = rng.random(n_prueba) < ctr[anuncio]
    ganador = np.argmax([clics_prueba[anuncio == i].mean() for i in range(k)])
    clics_resto = rng.random(visitas - n_prueba) < ctr[ganador]
    return clics_prueba.sum() + clics_resto.sum(), ganador

totales, ganadores = zip(*[prueba_ab(CTR, VISITAS, semilla=s) for s in range(300)])
print(f"Prueba A/B: {np.mean(totales):.0f} clics en promedio")
print(f"Eligió el mejor anuncio (B) en el {np.mean(np.array(ganadores) == 1):.0%} de los casos")

<details><summary><b>👀 Ver respuesta</b></summary>

Durante la prueba el CTR medio es (3 + 5 + 4.5 + 2) / 4 ≈ 3.6 %. Después muestra el ganador.

Elige B en **solo ~3 de cada 4 casos** (≈ 76 %): con 1,000 visitas por anuncio, la diferencia entre 5.0 % y 4.5 % (unos 5 clics) se confunde a menudo con el azar. Los clics promedio quedan alrededor de **925**, por debajo de los 1,000 del oráculo.
</details>

**Paso 3 ▶️ — Los bandidos.** Reutilizamos la función `jugar` de la lección (sirve igual: un clic es un "premio").

In [ ]:
for nombre, kwargs in [("ε-greedy ε=0.1      ", dict(epsilon=0.1)),
                       ("ε-greedy con decaim.", dict(epsilon=1.0, decaimiento=0.999))]:   # ✏️ prueba otros valores
    resultados = [jugar(n_jugadas=VISITAS, probs=CTR, semilla=s, **kwargs) for s in range(50)]   # 50 meses simulados (tarda ~20 s)
    clics = np.mean([p.sum() for _, p, _, _ in resultados])
    eligio_b = np.mean([(e[-2000:] == 1).mean() for e, _, _, _ in resultados])
    print(f"{nombre} → {clics:.0f} clics · en las últimas 2,000 visitas mostró el B el {eligio_b:.0%} del tiempo")

✋ Compara con la prueba A/B. ¿Qué estrategia consigue más clics?

✍️ *Mi comparación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con estas semillas:
- **ε = 0.1 fijo:** ≈ 930 clics, prácticamente igual que la prueba A/B (≈ 925). Sigue "regalando" el 10 % del tráfico a anuncios al azar todo el mes.
- **Con decaimiento (0.999):** ≈ 950 clics, la mejor de las tres.

Ninguna llega al oráculo (1,000): **explorar siempre cuesta algo**. Lo interesante es *cuánto* cuesta cada estrategia. Con diferencias tan pequeñas entre anuncios (5.0 % vs. 4.5 %), todas sufren para distinguir B de C.
</details>

**Paso 4 ✏️ — Ajusta tu estrategia.** Cambia el `decaimiento` en la celda del Paso 3 (prueba 0.9995 y 0.99). ¿Mejora o empeora? ¿Por qué crees que pasa?

✍️ *Mis resultados:*

<details><summary><b>👀 Ver respuesta</b></summary>

- **0.99**: ε cae casi a 0 en unas 500 visitas. Deja de explorar demasiado pronto y muchas veces se queda con C o con A: ≈ 905 clics, **peor** que la prueba A/B.
- **0.9995**: explora más tiempo (ε ≈ 0.37 a las 2,000 visitas): ve más veces los anuncios malos al principio, y el resultado (≈ 945) es parecido al de 0.999.

No existe un valor mágico: el mejor decaimiento depende de cuántas visitas tienes y de qué tan parecidas son las opciones. Por eso, igual que con cualquier hiperparámetro, se elige **simulando o validando**, no a ojo.
</details>

**Paso 5 ✋ — Recomendación al gerente.** Escribe 3 frases para el gerente de marketing (que no sabe ML) recomendando una estrategia. Incluye: qué gana, qué arriesga y qué vigilarías.

✍️ *Mi recomendación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo:

> «Recomiendo usar un sistema que **muestre más el anuncio que va ganando mientras aprende**, empezando por repartirlos a todos y concentrándose poco a poco en el mejor. En nuestras simulaciones logra unos clics más que la prueba A/B tradicional, porque desperdicia menos visitas en anuncios malos. Como dos de los anuncios son casi igual de buenos, lo vigilaríamos semanalmente y mantendríamos una pequeña parte del tráfico (~2 %) probando las demás opciones por si los gustos de los clientes cambian.»
</details>

---
## 📚 Para profundizar
- 📘 **Libro:** Sutton y Barto, *Reinforcement Learning: An Introduction*, capítulo 2 (*Multi-armed Bandits*): ε-greedy, valores optimistas y UCB.
- 📰 **Lectura:** Lattimore y Szepesvári, *Bandit Algorithms* (gratis en línea), para quien quiera ir a fondo.
- 🎮 **Interactivo:** busca "multi-armed bandit simulator" para jugar con distintas estrategias en el navegador.
- 💼 **Aplicación:** muchas plataformas de experimentación (Optimizely, Google Ads, VWO) ofrecen modos "bandido" además de las pruebas A/B clásicas.

---
⬅️ **Anterior:** [14.1 Agente, entorno, acción, recompensa: aprender a base de premios](14.1-agente-entorno-accion-recompensa.ipynb) · ➡️ **Siguiente:** [14.3 Q-learning y redes Q profundas: la chuleta del agente](14.3-q-learning-y-dqn.ipynb)